# Rol 1 — Analista de Incertidumbre e Información
### Proyecto Retador · UrbanMove · ML No Supervisado 2026-II · Universidad de La Sabana

**Pregunta de negocio:** ¿Qué tan predecible es la demanda de viajes por zona y momento del día? ¿Cuánto se gana (en bits) al usar información temporal para anticipar dónde ocurrirá la demanda?

## 1. Carga, limpieza y muestra estándar del proyecto

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.special import logsumexp

from src.utils import DEFAULT_TRAIN, add_features, clean_trips

raw = pd.read_csv(DEFAULT_TRAIN, parse_dates=["pickup_datetime"])
print(f"Registros totales: {len(raw):,}")

universe = clean_trips(add_features(raw))   # imprime filas removidas por regla

N_SAMPLE = 40_000
SEED = 42
df = universe.sample(n=N_SAMPLE, random_state=SEED).reset_index(drop=True)
print(f"Universo válido: {len(universe):,}  |  Muestra: {df.shape}")

Registros totales: 1,458,644
inside NYC bounding box             removed    633  (left 1458011)
duration between 1 min and 3 h      removed  10595  (left 1447416)
distance >= 0.1 km                  removed   7463  (left 1439953)
speed between 1 and 100 km/h        removed   1342  (left 1438611)
Total kept: 1438611/1458644 (98.6%)
Universo válido: 1,438,611  |  Muestra: (40000, 16)


**Base analítica común.** Se reproduce el mismo pipeline de datos del equipo: `train.csv` completo (1,458,644 registros) → variables derivadas y limpieza con `src.utils` (`add_features`, `clean_trips`) → **universo de 1,438,611 viajes válidos (98.6%)** → muestra aleatoria de **n = 40,000** con `SEED = 42`. No se aplica ningún filtro adicional: todos los cálculos de este notebook usan exactamente los 40,000 viajes de la muestra.

**Justificación del muestreo:** procesar el archivo completo en cada iteración es innecesario; con n = 40,000 el margen de error (95%) de una proporción p = 0.01 es ±0.098 puntos porcentuales, suficiente para estimar entropías sin ruido de muestreo relevante.

In [2]:
# Verificaciones de la muestra estándar del proyecto
print(f"Nulos en la muestra: {int(df[['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude','trip_duration','hour']].isna().sum().sum())}")
print(f"Rango: {df.pickup_datetime.min():%Y-%m-%d} a {df.pickup_datetime.max():%Y-%m-%d}")
print(f"Media distance_km = {df.distance_km.mean():.6f} | Media trip_duration = {df.trip_duration.mean():.6f}")

# 'hour' llega como float (hora + minutos/60); para variables discretas se usa la hora entera
df["hour_int"] = np.floor(df["hour"]).astype(int)
print("Horas discretas:", df.hour_int.min(), "a", df.hour_int.max())

Nulos en la muestra: 0
Rango: 2016-01-01 a 2016-06-30
Media distance_km = 3.454794 | Media trip_duration = 837.796275
Horas discretas: 0 a 23


**Verificación.** La muestra no tiene nulos en las variables usadas y cubre 2016-01-01 a 2016-06-30 (enero–junio de 2016). Sus medias (distancia = 3.455 km; duración = 837.8 s) sirven de huella para comprobar que se trabaja con la misma muestra. `hour` llega como número decimal (hora + minutos/60); para las variables discretas se usa su parte entera `hour_int` (0–23).

**Zonificación espacial:** se discretiza el espacio de recogida en una grilla de `0.02°` (~1.7×2.2 km por celda), resolución que separa los hotspots sin generar una cola de zonas casi vacías.

In [3]:
STEP = 0.02

def zone(lon, lat, step=STEP):
    return (np.floor(lon/step)).astype(int).astype(str) + "_" + (np.floor(lat/step)).astype(int).astype(str)

df["pickup_zone"] = zone(df.pickup_longitude, df.pickup_latitude)
df["dropoff_zone"] = zone(df.dropoff_longitude, df.dropoff_latitude)
print(f"Zonas de recogida ocupadas: {df.pickup_zone.nunique()}")

zc = df.pickup_zone.value_counts()
print(f"Zonas con < 10 viajes: {(zc < 10).sum()}  |  Zonas con >= 100 viajes: {(zc >= 100).sum()}")

Zonas de recogida ocupadas: 92
Zonas con < 10 viajes: 47  |  Zonas con >= 100 viajes: 22


Resultado: **92 zonas de recogida ocupadas**. La distribución es muy desigual: 47 zonas tienen menos de 10 viajes y solo 22 superan los 100. Las zonas raras aportan poca masa de probabilidad, por lo que las entropías son robustas a ellas, pero sus proporciones individuales son poco fiables.

## 2. Entropía de Shannon — ¿qué tan caótica es la demanda?

$$H(X) = -\sum_i p(x_i)\log_2 p(x_i)$$

Se calcula sobre la distribución empírica de viajes por zona de recogida (global, y condicionada a hora pico —7-9 h y 17-19 h— vs. madrugada —0-5 h—).

In [4]:
def entropy_bits(counts):
    p = np.asarray(counts, dtype=float)
    p = p / p.sum()
    p = p[p > 0]  # evita log(0)
    return -np.sum(p * np.log2(p))

n_zones = len(zc)
H_zone = entropy_bits(zc.values)
H_max = np.log2(n_zones)
redundancy = 1 - H_zone / H_max
top3 = zc.head(3).sum() / zc.sum()

print(f"Zonas activas: {n_zones}")
print(f"H(zona) = {H_zone:.3f} bits   |   H_max (uniforme) = {H_max:.3f} bits")
print(f"Redundancia = {redundancy:.1%}")
print(f"Zonas 'efectivas' (2^H) = {2**H_zone:.1f} de {n_zones} reales")
print(f"Participación de las 3 zonas top: {top3:.1%}")

Zonas activas: 92
H(zona) = 3.931 bits   |   H_max (uniforme) = 6.524 bits
Redundancia = 39.7%
Zonas 'efectivas' (2^H) = 15.3 de 92 reales
Participación de las 3 zonas top: 40.6%


**Lectura de negocio:** con 92 zonas activas, la incertidumbre uniforme sería de 6.524 bits. La demanda real tiene **H(zona) = 3.931 bits**, una redundancia del **39.7%**: se comporta como si hubiera solo **~15.3 zonas igualmente probables** (2^H), no 92. Las 3 zonas más activas concentran el **40.6%** de los viajes.

**Implicación operativa:** UrbanMove no necesita distribuir la flota de forma uniforme; pocas celdas explican la mayoría de la demanda.

In [5]:
PEAK_H, VALLEY_H = [7, 8, 9, 17, 18, 19], [0, 1, 2, 3, 4, 5]
peak = df[df.hour_int.isin(PEAK_H)]        # horas pico AM/PM
valley = df[df.hour_int.isin(VALLEY_H)]    # madrugada

H_peak = entropy_bits(peak.pickup_zone.value_counts().values)
H_valley = entropy_bits(valley.pickup_zone.value_counts().values)

print(f"H(zona | hora pico)   = {H_peak:.3f} bits   (n={len(peak):,})")
print(f"H(zona | madrugada)   = {H_valley:.3f} bits   (n={len(valley):,})")

top5_peak = peak.pickup_zone.value_counts(normalize=True).head(5).sum()
top5_valley = valley.pickup_zone.value_counts(normalize=True).head(5).sum()
print(f"Top-5 zonas cubren {top5_peak:.1%} de la demanda pico y {top5_valley:.1%} de la de madrugada")

H(zona | hora pico)   = 3.887 bits   (n=12,224)
H(zona | madrugada)   = 4.025 bits   (n=4,790)
Top-5 zonas cubren 59.5% de la demanda pico y 61.2% de la de madrugada


In [6]:
# ¿La diferencia H_pico - H_madrugada es real o ruido de muestreo? Bootstrap (500 remuestreos)
rng = np.random.default_rng(SEED)
codes = {name: pd.factorize(sub.pickup_zone)[0] for name, sub in [("pico", peak), ("madrugada", valley)]}

def boot_H(c, B=500):
    out = np.empty(B)
    for b in range(B):
        out[b] = entropy_bits(np.bincount(rng.choice(c, size=len(c), replace=True)))
    return out

bp, bv = boot_H(codes["pico"]), boot_H(codes["madrugada"])
diff = bv - bp
print(f"IC95% H(pico)      = [{np.percentile(bp, 2.5):.3f}, {np.percentile(bp, 97.5):.3f}]")
print(f"IC95% H(madrugada) = [{np.percentile(bv, 2.5):.3f}, {np.percentile(bv, 97.5):.3f}]")
print(f"Diferencia H(madrugada) - H(pico) = {H_valley - H_peak:.3f} bits | IC95% = [{np.percentile(diff, 2.5):.3f}, {np.percentile(diff, 97.5):.3f}]")

IC95% H(pico)      = [3.858, 3.911]
IC95% H(madrugada) = [3.959, 4.064]
Diferencia H(madrugada) - H(pico) = 0.138 bits | IC95% = [0.070, 0.187]


**Lectura de negocio:** la entropía condicionada baja a **3.887 bits** en hora pico frente a **4.025 bits** en madrugada: la demanda pico está más concentrada geográficamente. Las 5 zonas top cubren 59.5% de la demanda pico y 61.2% de la de madrugada.

**Cautela estadística:** la diferencia (0.138 bits) se contrasta con bootstrap (celda siguiente): el IC95% de la diferencia es [0.070, 0.187] bits y **excluye el cero**, así que la diferencia es real, pero modesta. La madrugada tiene solo n = 4,790 viajes, por lo que su entropía es más incierta (IC más ancho). **Decisión concreta:** priorizar cobertura densa en las zonas top durante la hora pico y mantener cobertura más amplia, con menos vehículos por celda, en la madrugada.

## 3. Entropía cruzada y Divergencia KL — día laboral vs. fin de semana

$$H(P,Q) = -\sum_i p(x_i)\log_2 q(x_i) \qquad D_{KL}(P\|Q) = \sum_i p(x_i)\log_2\frac{p(x_i)}{q(x_i)} = H(P,Q) - H(P)$$

$P$ = distribución de zona de recogida en día laboral, $Q$ = distribución en fin de semana. Se usa suavizado aditivo (+0.5) para evitar $q(x_i)=0$ en zonas sin observaciones en alguno de los dos contextos.

In [7]:
def dist_over_zones(sub, all_zones, alpha=0.5):
    c = sub.pickup_zone.value_counts().reindex(all_zones, fill_value=0).values.astype(float)
    c = c + alpha
    return c / c.sum()

all_zones = df.pickup_zone.unique()
wd, we = df[df.is_weekend == 0], df[df.is_weekend == 1]

P = dist_over_zones(wd, all_zones)   # laboral
Q = dist_over_zones(we, all_zones)   # fin de semana

H_P = -np.sum(P * np.log2(P))
H_cross = -np.sum(P * np.log2(Q))
KL_PQ = np.sum(P * np.log2(P / Q))
KL_QP = np.sum(Q * np.log2(Q / P))

print(f"n laboral={len(wd):,}  n fin de semana={len(we):,}  zonas={len(all_zones)}")
print(f"H(laboral) = {H_P:.4f} bits")
print(f"H(laboral, finde) [cruzada] = {H_cross:.4f} bits")
print(f"D_KL(laboral || finde) = {KL_PQ:.4f} bits")
print(f"D_KL(finde || laboral) = {KL_QP:.4f} bits   (confirma asimetría de la KL)")

n laboral=28,481  n fin de semana=11,519  zonas=92
H(laboral) = 3.9018 bits
H(laboral, finde) [cruzada] = 3.9403 bits
D_KL(laboral || finde) = 0.0385 bits
D_KL(finde || laboral) = 0.0442 bits   (confirma asimetría de la KL)


In [8]:
# Referencia de "ruido puro": KL entre dos mitades aleatorias de la muestra (sin diferencia real por construcción)
rng = np.random.default_rng(SEED)
kl_null = []
for _ in range(200):
    lab = rng.permutation(len(df)) < len(wd)          # mismas proporciones n_laboral / n_finde
    p_ = dist_over_zones(df[lab], all_zones); q_ = dist_over_zones(df[~lab], all_zones)
    kl_null.append(np.sum(p_ * np.log2(p_ / q_)))
kl_null = np.array(kl_null)
print(f"KL bajo H0 (etiquetas permutadas): media = {kl_null.mean():.4f}, máx = {kl_null.max():.4f} bits")
print(f"KL observada = {KL_PQ:.4f} bits  ->  {KL_PQ / kl_null.mean():.1f}x la media nula")

KL bajo H0 (etiquetas permutadas): media = 0.0062, máx = 0.0085 bits
KL observada = 0.0385 bits  ->  6.2x la media nula


**Lectura de negocio:** $D_{KL}(\text{laboral}\|\text{finde}) = 0.0385$ bits, asimétrica respecto a $D_{KL}(\text{finde}\|\text{laboral}) = 0.0442$ bits (KL no es una distancia). Codificar la demanda laboral con el modelo de fin de semana cuesta 3.9403 bits frente a 3.9018 bits del modelo propio: un sobrecosto de **0.0385 bits por viaje**.

**¿Es señal o ruido?** Con etiquetas laboral/finde permutadas al azar (mismos tamaños, 200 repeticiones) la KL promedio es 0.0062 bits (máx. 0.0085); la KL observada es **6.2 veces** la media nula y supera el máximo de las permutaciones. La diferencia entre contextos es real aunque pequeña en términos absolutos: apoya evaluar **dos políticas de posicionamiento** (laboral / fin de semana), con impacto acotado en la asignación por zona.

## 4. Información mutua y entropía condicional — ¿cuánto dice la hora sobre la zona?

$$I(H;Z) = \sum_{h,z} p(h,z)\log_2\frac{p(h,z)}{p(h)p(z)} \qquad H(Z\mid H) = H(Z) - I(H;Z)$$

Se restringe a las 20 zonas con más demanda para tener una tabla de contingencia hora×zona (24×20) con soporte suficiente en cada celda.

In [9]:
top_zones = zc.head(20).index
sub = df[df.pickup_zone.isin(top_zones)].copy()

joint = pd.crosstab(sub["hour_int"], sub["pickup_zone"])
Pxy = joint.values / joint.values.sum()
Px = Pxy.sum(axis=1, keepdims=True)
Py = Pxy.sum(axis=0, keepdims=True)
mask = Pxy > 0
MI = np.sum(Pxy[mask] * np.log2(Pxy[mask] / (Px @ Py)[mask]))

H_hour = entropy_bits(sub["hour_int"].value_counts().values)
H_zone20 = entropy_bits(sub["pickup_zone"].value_counts().values)
H_zone_given_hour = H_zone20 - MI

print(f"n (top-20 zonas) = {len(sub):,}  ({len(sub)/len(df):.1%} de la muestra)")
print(f"H(hora) = {H_hour:.4f} bits")
print(f"H(zona | top20) = {H_zone20:.4f} bits")
print(f"I(hora ; zona) = {MI:.4f} bits")
print(f"H(zona | hora) = {H_zone_given_hour:.4f} bits")
print(f"Reducción de incertidumbre = {MI/H_zone20:.1%}")

n (top-20 zonas) = 38,654  (96.6% de la muestra)
H(hora) = 4.4571 bits
H(zona | top20) = 3.6797 bits
I(hora ; zona) = 0.0527 bits
H(zona | hora) = 3.6270 bits
Reducción de incertidumbre = 1.4%


In [10]:
# El estimador plug-in de I(H;Z) tiene sesgo positivo con n finito: se compara con hora permutada (I = 0 real)
def mi_bits(h, z):
    J = pd.crosstab(h, z).values / len(h)
    px, py = J.sum(1, keepdims=True), J.sum(0, keepdims=True)
    m = J > 0
    return np.sum(J[m] * np.log2(J[m] / (px @ py)[m]))

rng = np.random.default_rng(SEED)
mi_null = np.array([mi_bits(rng.permutation(sub["hour_int"].values), sub["pickup_zone"].values) for _ in range(200)])
print(f"I(hora;zona) bajo independencia (permutación): media = {mi_null.mean():.4f}, máx = {mi_null.max():.4f} bits")
print(f"I observada = {MI:.4f} bits  |  I corregida por sesgo = {MI - mi_null.mean():.4f} bits ({(MI - mi_null.mean())/H_zone20:.1%} de H(zona))")

I(hora;zona) bajo independencia (permutación): media = 0.0083, máx = 0.0099 bits
I observada = 0.0527 bits  |  I corregida por sesgo = 0.0444 bits (1.2% de H(zona))


**Lectura de negocio:** $I(\text{hora};\text{zona}) = 0.0527$ bits, **1.4%** de la entropía de zona entre las top-20 (que cubren 96.6% de la muestra). Como el estimador con n finito tiene sesgo positivo, se verifica con hora permutada: bajo independencia real da 0.0083 bits (máx. 0.0099); la información observada está muy por encima, y corregida por sesgo queda en **0.0444 bits (1.2%)**.

Conocer la hora reduce poco la incertidumbre sobre *cuál* será la zona: la ubicación de los hotspots es estructural. Lo que sí varía con la hora es *cuánta* demanda hay en ellos.

## 5. Desigualdad de Jensen y *log-sum-exp trick*

La **desigualdad de Jensen** ($\mathbb{E}[\log X] \le \log \mathbb{E}[X]$, log cóncavo) implica $H(P,Q)\ge H(P)$, es decir $D_{KL}\ge 0$, y es la base de que EM maximice una **cota inferior** de la log-verosimilitud. Aquí se verifica numéricamente con las distribuciones $P$, $Q$ de la Sección 3.

In [11]:
# Jensen (log cóncavo): E_P[log2(Q/P)] <= log2 E_P[Q/P] = log2(sum Q) = 0  =>  D_KL(P||Q) >= 0
lhs = np.sum(P * np.log2(Q / P))     # = -D_KL(P||Q)
rhs = np.log2(np.sum(P * (Q / P)))   # = log2(sum Q) = 0
print(f"E_P[log2(Q/P)] = {lhs:.4f}  <=  log2 E_P[Q/P] = {rhs:.4f}")
print(f"H(P,Q) - H(P) = {H_cross - H_P:.4f} bits >= 0  (= D_KL(P||Q))")

E_P[log2(Q/P)] = -0.0385  <=  log2 E_P[Q/P] = 0.0000
H(P,Q) - H(P) = 0.0385 bits >= 0  (= D_KL(P||Q))


**Resultado:** $\mathbb{E}_P[\log_2(Q/P)] = -0.0385 \le \log_2\mathbb{E}_P[Q/P] = 0.0000$, y $H(P,Q)-H(P) = 0.0385 \ge 0$: la desigualdad se cumple con los datos propios.

**Log-sum-exp con datos propios.** Se construye una mezcla de 24 modelos de zona, uno por hora, $p_h(z)$ con pesos $\pi_h = P(\text{hora}=h)$. La log-verosimilitud de toda la muestra bajo cada modelo es $\ell_h=\sum_i \log p_h(z_i)$, y la de la mezcla requiere $\log\sum_h \exp(\log\pi_h + \ell_h)$, la misma estructura del denominador del paso E de un GMM. Es una demostración de escala numérica, no un modelo final.

In [12]:
# 24 modelos de zona por hora, p_h(z) (suavizado +0.5), con pesos pi_h = P(hora = h).
# log-verosimilitud de TODA la muestra bajo cada modelo: l_h = sum_i log p_h(z_i)
# log-verosimilitud de la mezcla: log sum_h pi_h * exp(l_h)  (misma estructura del denominador del paso E)
zones_idx = pd.Index(all_zones)
z_i = zones_idx.get_indexer(df.pickup_zone)
cnt_hz = pd.crosstab(df.hour_int, df.pickup_zone).reindex(columns=zones_idx, fill_value=0).values + 0.5
log_p_hz = np.log(cnt_hz / cnt_hz.sum(axis=1, keepdims=True))        # (24, zonas)
log_pi = np.log(np.bincount(df.hour_int, minlength=24) / len(df))    # (24,)

l_h = log_p_hz[:, z_i].sum(axis=1)                                   # (24,)
scaled = log_pi + l_h                                                # log(pi_h) + l_h

m = scaled.max()
logsumexp_stable = m + np.log(np.sum(np.exp(scaled - m)))

with np.errstate(over="ignore", under="ignore", divide="ignore"):
    naive = np.log(np.sum(np.exp(scaled)))

print(f"Rango de log(pi_h) + l_h: [{scaled.min():.1f}, {scaled.max():.1f}]")
print(f"log-sum-exp (estable) = {logsumexp_stable:.3f}")
print(f"scipy.special.logsumexp = {logsumexp(scaled):.3f}")
print(f"Cálculo ingenuo log(sum(exp(x))) = {naive}")
print(f"exp(x) == 0 en {(np.exp(scaled) == 0).sum()} de {len(scaled)} términos")

Rango de log(pi_h) + l_h: [-124344.2, -109686.6]
log-sum-exp (estable) = -109686.645
scipy.special.logsumexp = -109686.645
Cálculo ingenuo log(sum(exp(x))) = -inf
exp(x) == 0 en 24 de 24 términos


**Resultado:** los 24 términos $\log\pi_h+\ell_h$ están entre -124344.2 y -109686.6. Al exponenciar directamente, `exp(x)` da `0.0` en **24 de 24** términos y `log(0) = -inf`. Restando el máximo antes de exponenciar, el log-sum-exp estable da **-109686.645**, idéntico a `scipy.special.logsumexp`.

## 6. Síntesis — insumos cuantitativos para el equipo

Todas las cifras provienen de la muestra estándar de **40,000 viajes** (`SEED = 42`, extraída del universo de 1,438,611 viajes válidos) y de la zonificación `STEP = 0.02°` (92 zonas).

| Métrica | Valor | Detalle | Uso aguas abajo |
|---|---|---|---|
| H(zona) global | 3.931 bits | 39.7% redundancia; 15.3 zonas efectivas de 92 | Rol 2: la demanda se concentra en pocas zonas |
| H(zona\|pico) vs H(zona\|madrugada) | 3.887 vs 4.025 bits | Δ = 0.138 bits, IC95% [0.070, 0.187] | Rol 4: redistribución horaria de flota |
| D_KL(laboral‖finde) | 0.0385 bits | inversa 0.0442; 6.2× el nivel nulo | Rol 4: dos políticas operativas |
| I(hora;zona), top-20 | 0.0527 bits | 1.4% de H(zona); 0.0444 bits corregida por sesgo | Rol 3: la hora no determina la zona, conviene mezcla probabilística |
| log-sum-exp | -109686.645 | el cálculo ingenuo da -inf | Rol 3: estabilidad numérica en el paso E |

In [13]:
summary = pd.DataFrame([
    ("H(zona) global", f"{H_zone:.3f} bits", f"{redundancy:.1%} redundancia vs. uniforme; {2**H_zone:.1f} zonas efectivas de {n_zones}"),
    ("H(zona | pico) / H(zona | madrugada)", f"{H_peak:.3f} / {H_valley:.3f} bits", f"Δ = {H_valley - H_peak:.3f} bits (IC95% [{np.percentile(diff, 2.5):.3f}, {np.percentile(diff, 97.5):.3f}])"),
    ("D_KL(laboral || finde)", f"{KL_PQ:.4f} bits", f"inversa {KL_QP:.4f}; {KL_PQ / kl_null.mean():.1f}x el nivel nulo ({kl_null.mean():.4f})"),
    ("I(hora ; zona), top-20", f"{MI:.4f} bits", f"{MI/H_zone20:.1%} de H(zona|top20); corregida por sesgo {MI - mi_null.mean():.4f} bits"),
    ("log-sum-exp", f"{logsumexp_stable:.3f}", "ingenuo = -inf"),
], columns=["Métrica", "Valor", "Detalle"])
summary

,Métrica,Valor,Detalle
0,H(zona) global,3.931 bits,39.7% redundancia vs. uniforme; 15.3 zonas efe...
1,H(zona | pico) / H(zona | madrugada),3.887 / 4.025 bits,"Δ = 0.138 bits (IC95% [0.070, 0.187])"
2,D_KL(laboral || finde),0.0385 bits,inversa 0.0442; 6.2x el nivel nulo (0.0062)
3,"I(hora ; zona), top-20",0.0527 bits,1.4% de H(zona|top20); corregida por sesgo 0.0...
4,log-sum-exp,-109686.645,ingenuo = -inf
